# Databricks Platform Lab 

## Quick Concept Map: Snowflake → Databricks

| Snowflake Concept | Databricks Equivalent | Notes |
|---|---|---|
| Account / Organization | Unity Catalog Metastore | Single governance layer across workspaces |
| Database.Schema | Catalog.Schema | Three-level namespace: `catalog.schema.object` |
| Role (RBAC) | Groups + Grants + ABAC Policies | No role hierarchy; use groups + attribute-based policies |
| Warehouse (XS–6XL) | SQL Warehouse / Job Cluster | SQL Warehouses for BI; Job Clusters for ETL |
| Clustering Keys | Liquid Clustering (`CLUSTER BY AUTO`) | Replaces Z-ORDER; automatic, incremental |
| Tasks / DAGs | Lakeflow Jobs (formerly Workflows) | Multi-task DAGs with dependencies |
| Streams + Tasks | Lakeflow Spark Declarative Pipelines (SDP) | Streaming tables + materialized views |
| Secure Data Sharing | Open Sharing | Open protocol, previously known as Delta Sharing; cross-platform (Snowflake, Spark, pandas) |
| Stages | Unity Catalog Volumes + External Locations | Managed or external file storage |
| Terraform / CLI | Terraform / CLI  / Declarative Automation Bundles (DABs) | IaC for notebooks, pipelines, jobs |

---

## Lab Setup: Naming Convention

Throughout this lab, replace these placeholders with your actual values:
- `<YOUR_CATALOG>` — your sandbox catalog
- `<YOUR_SCHEMA>` — your lab schema
- `<YOUR_USER>` — your Databricks username (email)
- `<YOUR_STORAGE_CREDENTIAL>` — credential name for external storage
- `<YOUR_EXTERNAL_LOCATION>` — path to external cloud storage

# Section 1 — Unity Catalog: Identity, Grants & Security

## Databricks Governance Model

-  No roles — use **Groups** (synced from IdP via SCIM) 
- `GRANT SELECT ON TABLE` to a group or user directly 
-  Flat hyerarchy: each grant is explicit; catalog/schema owners inherit 
-  **ABAC Row Filter Policies** (tag-driven, inherited)  
-  **ABAC Column Mask Policies** (tag-driven, inherited)  


In [0]:
%sql
-- ============================================================
-- 1.1  CREATE A SANDBOX CATALOG AND SCHEMA
-- ============================================================
-- In Databricks, the top-level container is a CATALOG.

CREATE CATALOG IF NOT EXISTS lab_snowflake_admin;

CREATE SCHEMA IF NOT EXISTS lab_snowflake_admin.governance_lab
  COMMENT 'Lab schema for testing UC governance features';

In [0]:
%sql
-- ============================================================
-- 1.2  IDENTITY: GROUPS & GRANTS
-- ============================================================
--
-- NOTE: Group-specific grants are at the BOTTOM of this cell.
-- They will only succeed AFTER cell 1.2a creates the groups.
-- ============================================================

-- Step 1: Grant schema-level access to all users (like GRANT USAGE ON SCHEMA)
GRANT USE SCHEMA ON SCHEMA lab_snowflake_admin.governance_lab TO `account users`;
GRANT SELECT ON SCHEMA lab_snowflake_admin.governance_lab TO `account users`;

-- Step 2: View current grants (like SHOW GRANTS ON SCHEMA in Snowflake)
SHOW GRANTS ON SCHEMA lab_snowflake_admin.governance_lab;

## Row level access control and column masking using ABAC

To test the fine grained access control we will use an existing demo that can be installed in any workspace [Table ACL & Row and Column Level Security With Unity Catalog](https://www.databricks.com/resources/demos/tutorials/governance/table-acl-and-dynamic-views-with-uc)

##### Note you will need to create some additional groups using the UI , alternatively the CLI could be used from your laptop

In [0]:
%pip install dbdemos

In [0]:
import dbdemos
dbdemos.install('uc-01-acl')

## External Storage: Storage Credentials & External Locations

To test the creation, modification of external storage you need access to AWS console to create an IAM role with the privilege to manage the buckets on behalf of databricks, this is out of scope of today workshop. We will use the existing demo [Access data on External Location](https://www.databricks.com/resources/demos/tutorials/governance/access-data-on-external-location)

##### You can review the [notebook online](https://notebooks.databricks.com/demos/uc-02-external-location/index.html) before installing it in the cell below

In [0]:
import dbdemos
dbdemos.install('uc-02-external-location')

# Section 1b — Lakehouse Federation: Snowflake Connection & Foreign Catalog
Key Insight: Lakehouse Federation lets you query your Snowflake data directly from Databricks without copying it. Filters and aggregations are pushed down to Snowflake for performance.

In [0]:
%sql
-- ============================================================
-- 8.1  CREATE SNOWFLAKE CONNECTION (Basic Auth)
-- ============================================================
-- In Snowflake: You query locally — no "connection" concept needed.
-- In Databricks: CREATE CONNECTION defines how to reach Snowflake,
--               then CREATE FOREIGN CATALOG exposes it as a UC catalog.
-- ============================================================

CREATE CONNECTION IF NOT EXISTS lab_snowflake_fed
TYPE snowflake
OPTIONS (
  host 'MINVHHR-XAB68878.snowflakecomputing.com',
  port '443',
  sfWarehouse 'COMPUTE_WH',
  user 'WITTYGNOM',
  password 'Sn0wt3st202606',
  use_proxy 'false'
);

-- Verify the connection
SHOW CONNECTIONS;

In [0]:
%sql
-- ============================================================
-- 8.2  CREATE FOREIGN CATALOG (Snowflake → Unity Catalog)
-- ============================================================
-- This mirrors the Snowflake database 'ml_workshop' into UC
-- as a federated catalog. You can then query:
--   SELECT * FROM lab_snowflake_federation.sales.<table_name>
--
-- In Snowflake: USE DATABASE ml_workshop; USE SCHEMA sales;
-- In Databricks: Three-part name via the foreign catalog.
-- ============================================================

CREATE FOREIGN CATALOG IF NOT EXISTS lab_snowflake_federation
USING CONNECTION lab_snowflake_fed
OPTIONS (database = 'ml_workshop');

-- Refresh metadata so schemas and tables are visible immediately
REFRESH FOREIGN CATALOG lab_snowflake_federation;

-- Verify: list schemas (should include 'sales')
SHOW SCHEMAS IN lab_snowflake_federation;

-- Query the sales schema
-- SELECT * FROM lab_snowflake_federation.sales.<table_name> LIMIT 10;

# Section 2 — Compute & Cost Management

## Databricks Compute Model

| Compute type | When to Use |
|---|---|
| **SQL Warehouse** (Serverless or Pro) | BI tools, ad-hoc SQL, dashboards |
| SQL Warehouse with auto-scaling | Concurrency scaling for many users |
| Serverless SQL (automatic) | Built into serverless; no config needed |
| SQL Warehouse auto-stop (5min–) | Cost control when idle |
| **Job Cluster** (ephemeral) | ETL/ML pipelines; spins up, runs, dies |
| **All-Purpose Cluster** | Interactive development (notebooks) |

## Resource Monitors 

| Tool | description |
|---|---|
| **Budgets + System Tables** | Cost alerting and tracking |


In [0]:
%sql
-- ============================================================
-- 2.1  SQL WAREHOUSES (Equivalent to Snowflake Virtual Warehouses)
-- ============================================================
-- Snowflake: CREATE WAREHOUSE analyst_wh WAREHOUSE_SIZE = 'MEDIUM' ...
-- Databricks: Managed via UI/API/DABs (not SQL DDL)

-- View available SQL warehouses (like SHOW WAREHOUSES in Snowflake)
SELECT 
  warehouse_id,
  warehouse_name,
  warehouse_type,      -- PRO, CLASSIC, or SERVERLESS
  warehouse_size,      -- Like Snowflake T-shirt sizes
  min_clusters,
  max_clusters,        -- Auto-scaling: like multi-cluster warehouses
  auto_stop_minutes,   -- Like AUTO_SUSPEND in Snowflake
  warehouse_channel
FROM system.compute.warehouses
WHERE delete_time IS NULL
ORDER BY warehouse_name;

In [0]:
# ============================================================
# 2.2  CREATE & MANAGE SQL WAREHOUSE PROGRAMMATICALLY
# ============================================================
# In Snowflake: CREATE WAREHOUSE ... ALTER WAREHOUSE ...
# In Databricks: Use the SDK or REST API (no SQL DDL for warehouses)

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import (
    CreateWarehouseRequestWarehouseType,
    SpotInstancePolicy,
    Channel,
    ChannelName,
)

w = WorkspaceClient()

# Example: Create a serverless SQL warehouse for BI analysts
# Equivalent to: CREATE WAREHOUSE bi_analysts SIZE='MEDIUM' MIN_CLUSTER_COUNT=1 MAX_CLUSTER_COUNT=3

warehouse = w.warehouses.create_and_wait(
    name='bi_analysts_lab',
    cluster_size='Medium',          # 2X-Small, X-Small, Small, Medium, Large, X-Large, 2X-Large, 3X-Large, 4X-Large
    min_num_clusters=1,             # Min clusters (like MIN_CLUSTER_COUNT in multi-cluster WH)
    max_num_clusters=3,             # Max clusters for auto-scaling
    auto_stop_mins=1,              # Auto-suspend after 10 min idle (Snowflake: AUTO_SUSPEND = 600)
    warehouse_type=CreateWarehouseRequestWarehouseType.PRO,  # PRO for SQL features; use SERVERLESS when available
    enable_serverless_compute=True, # Serverless = no infra management (like Snowflake default)
    channel=Channel(name=ChannelName.CHANNEL_NAME_CURRENT),
    tags=None
)
print(f"Created warehouse: {warehouse.id}")


# List existing warehouses
for wh in w.warehouses.list():
    print(f"  {wh.name:<30} | Size: {wh.cluster_size:<10} | State: {wh.state} | Clusters: {wh.min_num_clusters}-{wh.max_num_clusters}")

In [0]:
%sql
-- ============================================================
-- 2.3  COST MONITORING (Equivalent to Snowflake Resource Monitors)
-- ============================================================
-- Databricks: Query system.billing tables + set up Budgets in the account console

-- DBU consumption by warehouse (last 7 days)
-- Like: SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
SELECT 
  usage_metadata.warehouse_id,
  sku_name,
  SUM(usage_quantity) AS total_dbus,
  COUNT(DISTINCT usage_date) AS active_days,
  ROUND(SUM(usage_quantity) / COUNT(DISTINCT usage_date), 2) AS avg_dbus_per_day
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE - INTERVAL 7 DAYS
  AND usage_metadata.warehouse_id IS NOT NULL
GROUP BY 1, 2
ORDER BY total_dbus DESC
LIMIT 20;

-- ===========================================================================
-- For more labs on System tables install the demo 
-- [System Tables: Billing Forecast, Usage Analytics, and Access Auditing With Databricks Unity Catalog](https://www.databricks.com/resources/demos/tutorials/governance/system-tables)
-- dbdemos.install('uc-04-system-tables', catalog='cjc', schema='billing_forecast')

In [0]:
%sql
-- ============================================================
-- 2.4  QUERY PERFORMANCE MONITORING
-- ============================================================
-- Snowflake: QUERY_HISTORY view, INFORMATION_SCHEMA.QUERY_HISTORY()
-- Databricks: system.query.history

-- Slowest queries in the last 24 hours
SELECT 
  statement_id,
  executed_by,
  compute.warehouse_id,
  statement_type,
  ROUND(total_duration_ms / 1000.0, 2) AS duration_seconds,
  produced_rows,
  statement_text
FROM system.query.history
WHERE start_time >= CURRENT_TIMESTAMP - INTERVAL 24 HOURS
  AND execution_status = 'FINISHED'
ORDER BY total_duration_ms DESC
LIMIT 10;

In [0]:
# ============================================================
# 2.5  JOB CLUSTERS (No Snowflake Equivalent)
# ============================================================
# Job clusters are ephemeral: created when a job starts, destroyed when it ends.
# This is how you run production ETL/ML without paying for idle compute.
#
# Snowflake closest analog: A warehouse that auto-suspends immediately,
# but Job Clusters are cheaper (Spot instances) and more flexible (custom libs).

# Job cluster configuration (used in Lakeflow Jobs or DABs)
job_cluster_config = {
    "spark_version": "17.3.x-scala2.12",   # Latest LTS runtime
    "node_type_id": "i3.xlarge",            # Instance type (like warehouse size)
    "autoscale": {
        "min_workers": 1,                    # Like MIN_CLUSTER_COUNT
        "max_workers": 8                     # Like MAX_CLUSTER_COUNT
    },
    "aws_attributes": {
        "availability": "SPOT_WITH_FALLBACK", # Cost saving: use Spot instances
        "spot_bid_max_price": -1              # Auto-bid
    },
    "spark_conf": {
        "spark.databricks.delta.optimizeWrite.enabled": "true",
        "spark.databricks.adaptive.autoOptimizeShuffle.enabled": "true"
    },
    "custom_tags": {
        "team": "data-engineering",
        "cost_center": "DE-001"
    }
}

print("""
┌────────────────────────────────────────────────────────────┐
│  COMPUTE DECISION TREE                                     │
├────────────────────────────────────────────────────────────┤
│  BI/SQL/Dashboards  → SQL Warehouse (Serverless preferred)  │
│  Interactive dev     → All-Purpose Cluster (or Serverless)  │
│  Scheduled ETL/ML   → Job Cluster (ephemeral, spot)         │
│  Streaming           → Job Cluster or SDP Pipeline compute   │
│  Ad-hoc exploration → Serverless Notebook compute           │
└────────────────────────────────────────────────────────────┘
""")

# Section 3 — Managed Tables: Storage Optimization & Data Retention

## Unity Catalog Managed Tables: Delta AND Iceberg

In Databricks, **managed tables** are the standard.   
Databricks managed tables support **two open formats**:

| Format | Default? | When to Use | Notes |
|---|---|---|---|
| **Delta Lake** | Yes (default) | Most workloads, streaming, MERGE, CDF | Databricks-native, most features |
| **Apache Iceberg v3** | Optional | Interop with Trino/Spark OSS/Flink/Dremio | `USING ICEBERG` at CREATE time |

```sql
-- Default: creates a Delta table
CREATE TABLE my_catalog.my_schema.orders (...);

-- Explicitly choose Iceberg v3 for cross-engine interoperability
CREATE TABLE my_catalog.my_schema.orders (...) USING ICEBERG;
```

> In Databricks, BOTH Delta and Iceberg are first-class managed table formats under Unity Catalog.  
> The optimization features below (Liquid Clustering, OPTIMIZE, VACUUM) apply to **both formats**.

## Snowflake vs Databricks Storage Optimization

| Snowflake | Databricks Managed Tables | Notes |
|---|---|---|
| Micro-partitions (automatic) | **Liquid Clustering** (`CLUSTER BY AUTO`) | Replaces Z-ORDER; automatic, incremental |
| Clustering Keys (manual) | `CLUSTER BY (col1, col2)` | Manual seeding for known patterns |
| Search Optimization Service | **Data Skipping** (automatic with clustering) | Built-in; no add-on cost |
| `ALTER TABLE ... CLUSTER BY (col)` | `ALTER TABLE ... CLUSTER BY (col)` | Similar syntax! |
| Automatic Reclustering | `CLUSTER BY AUTO` | ML-driven key selection |
| Time Travel (90 days) | **Time Travel** (configurable retention) | Access any version via `VERSION AS OF` |
| Data Retention (1–90 days) | **VACUUM** (default 7 days, configurable) | Physically deletes old file versions |
| *N/A* | **OPTIMIZE** | Compacts small files into larger ones |
| *N/A* | **Predictive Optimization** | Automatic OPTIMIZE + VACUUM (serverless) |

> **Key Insight:** Liquid Clustering replaces both Z-ORDER and the old OPTIMIZE+ZORDER workflow.  
> `CLUSTER BY AUTO` is the recommended default for new tables — it learns from your query patterns  
> and adjusts clustering keys automatically. Z-ORDER still works but is considered legacy.  
> These features work on both Delta and Iceberg managed tables.

In [0]:
%sql
-- ============================================================
-- 3.1  MANAGED TABLES & LIQUID CLUSTERING
-- ============================================================
-- Snowflake equivalent: ALTER TABLE ... CLUSTER BY (date, region)
-- Databricks: Same concept, but incremental and ML-optimized.
--
-- MANAGED TABLE FORMATS:
--   • Delta (default) — best for Databricks-native workloads
--   • Iceberg v3       — best for cross-engine interop (Trino, Flink, etc.)
--   Both support Liquid Clustering, OPTIMIZE, VACUUM, Time Travel.

-- ================================================================
-- DELTA TABLE (default format — no USING clause needed)
-- ================================================================
-- Strategy A: You know the access patterns → Seed keys, then enable Auto
CREATE OR REPLACE TABLE lab_snowflake_admin.governance_lab.events (
  event_id BIGINT GENERATED ALWAYS AS IDENTITY,
  event_date DATE,
  user_id BIGINT,
  event_type STRING,
  region STRING,
  payload STRING
)
-- Manual clustering keys (like Snowflake CLUSTER BY)
CLUSTER BY (event_date, region);

-- Now enable automatic optimization on top (learns from query patterns)
ALTER TABLE lab_snowflake_admin.governance_lab.events CLUSTER BY AUTO;

-- Strategy B: No idea about patterns → Let the system figure it out
CREATE OR REPLACE TABLE lab_snowflake_admin.governance_lab.events_auto (
  event_id BIGINT GENERATED ALWAYS AS IDENTITY,
  event_date DATE,
  user_id BIGINT,
  event_type STRING,
  payload STRING
)
CLUSTER BY AUTO;  -- Pure automatic: system learns optimal keys

-- ================================================================
-- ICEBERG v3 TABLE (for cross-platform interoperability)
-- ================================================================
-- Use USING ICEBERG when consumers read via Trino, Flink, Spark OSS, etc.
-- Snowflake context: Snowflake supports reading Iceberg tables too,
-- so this enables bidirectional interop without Delta Sharing.
CREATE OR REPLACE TABLE lab_snowflake_admin.governance_lab.events_iceberg (
  event_id BIGINT,
  event_date DATE,
  user_id BIGINT,
  event_type STRING,
  region STRING,
  payload STRING
)
USING ICEBERG
TBLPROPERTIES (
  'delta.enableDeletionVectors' = false,
  'delta.enableRowTracking' = false
)
CLUSTER BY (event_date, region);  -- Liquid Clustering works on Iceberg too!

-- Verify clustering configuration (works for both formats)
DESCRIBE DETAIL lab_snowflake_admin.governance_lab.events;
DESCRIBE DETAIL lab_snowflake_admin.governance_lab.events_iceberg;

In [0]:
%sql
-- ============================================================
-- 3.2  OPTIMIZE (File Compaction)
-- ============================================================
-- Snowflake: Automatic (micro-partition rewrite is transparent)
-- Databricks: Explicit OPTIMIZE or enable Predictive Optimization.
-- Works on BOTH Delta and Iceberg managed tables.
--
-- WHY: Streaming and frequent small writes create many small files.
-- OPTIMIZE compacts them into larger, query-efficient files.

-- Insert sample data to demonstrate
INSERT INTO lab_snowflake_admin.governance_lab.events (event_date, user_id, event_type, region, payload)
SELECT 
  CURRENT_DATE - CAST(FLOOR(RAND() * 30) AS INT) AS event_date,
  CAST(FLOOR(RAND() * 10000) AS BIGINT) AS user_id,
  CASE CAST(FLOOR(RAND() * 4) AS INT)
    WHEN 0 THEN 'click'
    WHEN 1 THEN 'purchase'
    WHEN 2 THEN 'page_view'
    ELSE 'sign_up'
  END AS event_type,
  CASE CAST(FLOOR(RAND() * 3) AS INT)
    WHEN 0 THEN 'US'
    WHEN 1 THEN 'EU'
    ELSE 'APAC'
  END AS region,
  '{"source": "lab"}' AS payload
FROM RANGE(10000);

-- Run OPTIMIZE (compacts small files; also triggers Liquid Clustering)
-- In Snowflake, this happens automatically. In Databricks, run manually or enable Predictive Optimization.
OPTIMIZE lab_snowflake_admin.governance_lab.events;

-- View file metrics after optimization
DESCRIBE DETAIL lab_snowflake_admin.governance_lab.events;

In [0]:
%sql
-- ============================================================
-- 3.4  VACUUM (Data Retention & Storage Cleanup)
-- ============================================================
-- Snowflake equivalent: DATA_RETENTION_TIME_IN_DAYS (auto-managed)
-- Databricks: You control retention explicitly with VACUUM.
-- Works on both Delta and Iceberg managed tables.
--
-- VACUUM removes old file versions that are no longer needed for
-- time travel. Default retention: 7 days.

-- Check current table history (like Snowflake's Time Travel metadata)
DESCRIBE HISTORY lab_snowflake_admin.governance_lab.events LIMIT 10;

-- Time Travel: read an older version (like Snowflake: AT(TIMESTAMP => ...))
SELECT COUNT(*) AS row_count 
FROM lab_snowflake_admin.governance_lab.events VERSION AS OF 1;

-- Set retention period (default 7 days; Snowflake allows 0-90 days)
ALTER TABLE lab_snowflake_admin.governance_lab.events
  SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 7 days');

-- VACUUM: remove files older than retention period
-- DRY RUN first (like Snowflake's SYSTEM$ESTIMATE_DATA_EXPIRATION_SAVINGS)
VACUUM lab_snowflake_admin.governance_lab.events DRY RUN;

-- Actual cleanup (WARNING: removes time-travel ability for deleted versions)
-- VACUUM lab_snowflake_admin.governance_lab.events RETAIN 168 HOURS;

In [0]:
%sql
-- ============================================================
-- 3.5  PREDICTIVE OPTIMIZATION (Automatic OPTIMIZE + VACUUM)
-- ============================================================
-- This is the closest to Snowflake's fully automatic storage management.
-- When enabled, Databricks automatically runs OPTIMIZE and VACUUM
-- on your tables based on usage patterns. Requires serverless compute.

-- Enable at the schema level (applies to all tables in the schema)
ALTER SCHEMA lab_snowflake_admin.governance_lab
  ENABLE PREDICTIVE OPTIMIZATION;

-- Or enable/disable per table
ALTER TABLE lab_snowflake_admin.governance_lab.events
  ENABLE PREDICTIVE OPTIMIZATION;

-- Check optimization history
SELECT *
FROM system.storage.predictive_optimization_operations_history
WHERE table_name = 'events' AND schema_name = 'governance_lab'
ORDER BY start_time DESC
LIMIT 5;

# Section 4 — Orchestration: Lakeflow Spark Declarative Pipelines & Lakeflow Jobs

## What is SDP (Spark Declarative Pipelines)?

Lakeflow Spark Declarative Pipelines (SDP, formerly "Delta Live Tables") is a declarative ETL framework where you define *what* your data should look like — streaming tables, materialized views, and quality expectations — and the system automatically handles execution, dependency ordering, error recovery, and incremental processing. It replaces Snowflake Streams + Dynamic Tables in a single unified API.

## What are Lakeflow Jobs?

Lakeflow Jobs is Databricks' orchestration engine for scheduling and coordinating multi-task workflows as directed acyclic graphs (DAGs). Each task can run notebooks, pipelines, SQL, or Python scripts with dependencies, retries, conditional logic, and alerts — replacing Snowflake Task DAGs with a visual editor and ephemeral job clusters for cost efficiency.

---

## 🚀 How to Create an SDP Pipeline

You have **three options** to create a pipeline from the code in cells 4.1 (Python) and 4.2 (SQL) below:

### Option A: Use the Pipelines UI
1. Navigate to **Jobs & Pipelines → Create Pipeline** in the left sidebar
2. Choose "Spark Declarative Pipeline"
3. Add a new notebook to the pipeline (or point to an existing one)
4. Paste the code from **Cell 4.1** (Python API) or **Cell 4.2** (SQL syntax) into the pipeline notebook
5. Configure the target catalog/schema and compute settings
6. Click **Start** to run the pipeline

### Option B: Use Genie Code (AI Assistant) on the Pipeline Page
1. Navigate to **Jobs & Pipelines → Create Pipeline**
2. Open the AI assistant (Genie Code) in the pipeline editor
3. Describe your pipeline in natural language (e.g., *"Create a bronze-silver-gold pipeline that ingests JSON from a Volume, validates user_id and event_type, and aggregates daily summaries by region"*)
4. The assistant will generate the SDP code for you directly in the pipeline notebook

### Option C: Follow the Interactive Demos
* **[Lakeflow Declarative Pipeline Tutorial](https://www.databricks.com/resources/demos/tutorials/lakehouse-platform/lakeflow-declarative-pipeline)** — End-to-end walkthrough of building a medallion pipeline with SDP
* **[CDC Pipeline with SDP](https://www.databricks.com/resources/demos/tutorials/lakehouse-platform/cdc-pipeline-with-delta-live-table)** — Hands-on demo for Change Data Capture ingestion using AUTO CDC

---

## 🚀 How to Create a Lakeflow Job

For a hands-on lab building a multi-task job with SQL lookups, FOR EACH loops, and task dependencies, follow this tutorial:

* **[Lakeflow Jobs Hands-On Tutorial](https://docs.databricks.com/aws/en/jobs/how-to/foreach-sql-lookup-tutorial)** — Step-by-step guide to creating jobs with conditional logic and iteration

---

> **Key Insight:** SDP handles the *data transformation logic* (what to compute), while Lakeflow Jobs handle the *orchestration* (when and how to run it). You can embed an SDP pipeline as a task inside a Job for end-to-end automation.

In [0]:
# ============================================================
# 4.1  LAKEFLOW SPARK DECLARATIVE PIPELINE (SDP) - PYTHON API
# ============================================================
# This code runs INSIDE an SDP pipeline (not in a regular notebook).
# Save this as a separate notebook and attach it to a pipeline.
#
# NOTE: The old `import dlt` module is DEPRECATED.
#       The current API is: from pyspark import pipelines as dp
#       Decorators: @dp.table(), @dp.expect_or_drop(), etc.
#
# Snowflake equivalent:
#   CREATE STREAM raw_stream ON TABLE raw_events;
#   CREATE DYNAMIC TABLE silver_events AS SELECT ... FROM STREAM(raw_stream);
#   CREATE DYNAMIC TABLE gold_summary AS SELECT ... FROM silver_events;
#
# Can also be done entirely in SQL (see next cell 4.2).

# --- PASTE THIS INTO A PIPELINE NOTEBOOK ---

pipeline_code = '''
from pyspark import pipelines as dp  # Current API (replaces deprecated `import dlt`)
from pyspark.sql.functions import col, current_timestamp, year, month

# ---- BRONZE: Ingest raw data (like Snowpipe + Stream) ----
@dp.table(
    name="bronze_events",
    comment="Raw event ingestion via Auto Loader (like Snowpipe)",
    table_properties={"quality": "bronze"}
)
def bronze_events():
    return (
        spark.readStream
        .format("cloudFiles")       # Auto Loader (like Snowpipe)
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Auto schema evolution
        .load("/Volumes/lab_snowflake_admin/governance_lab/raw_files/events/")
    )

# ---- SILVER: Clean & validate (like Dynamic Table with DQ) ----
@dp.table(
    name="silver_events",
    comment="Cleaned events with quality expectations",
    table_properties={"quality": "silver"}
)
@dp.expect_or_drop("valid_user_id", "user_id IS NOT NULL AND user_id > 0")
@dp.expect_or_warn("valid_event_type", "event_type IN (\'click\',\'purchase\',\'page_view\',\'sign_up\')")
@dp.expect_or_fail("valid_timestamp", "event_timestamp IS NOT NULL")
def silver_events():
    return (
        dp.read_stream("bronze_events")
        .withColumn("ingested_at", current_timestamp())
        .withColumn("event_year", year(col("event_date")))
        .withColumn("event_month", month(col("event_date")))
    )

# ---- GOLD: Business aggregates (like Dynamic Table for analytics) ----
@dp.table(
    name="gold_daily_summary",
    comment="Daily event summary for BI dashboards",
    table_properties={"quality": "gold"}
)
def gold_daily_summary():
    return (
        dp.read("silver_events")
        .groupBy("event_date", "event_type", "region")
        .agg(
            {"event_id": "count", "user_id": "approx_count_distinct"}
        )
        .withColumnRenamed("count(event_id)", "event_count")
        .withColumnRenamed("approx_count_distinct(user_id)", "unique_users")
    )
'''

print(pipeline_code)
print("\n" + "="*70)
print("NOTE: `from pyspark import pipelines as dp` is the CURRENT API.")
print("      `import dlt` still works but is deprecated.")
print("")
print("INSTRUCTIONS: Copy the above into a new notebook and attach it")
print("to a Lakeflow Spark Declarative Pipeline via the Pipelines UI or DABs.")
print("")
print("TIP: This same pipeline can be written entirely in SQL (see cell 4.2).")
print("="*70)

In [0]:
%sql
-- ============================================================
-- 4.2  SDP PIPELINE IN SQL (Declarative Syntax)
-- ============================================================
-- If you prefer SQL (like Snowflake Dynamic Tables), you can
-- define SDP pipelines entirely in SQL. This runs inside a pipeline.

-- PASTE INTO A PIPELINE SQL NOTEBOOK:

-- Bronze: Streaming table from Auto Loader
-- Snowflake equivalent: CREATE STREAM + Snowpipe
/*
CREATE OR REFRESH STREAMING TABLE bronze_orders
COMMENT 'Raw orders ingested from cloud storage'
AS SELECT *
FROM STREAM read_files(
  '/Volumes/lab_snowflake_admin/governance_lab/raw_files/orders/',
  format => 'csv',
  header => true,
  inferSchema => true
);

-- Silver: Materialized View with expectations
-- Snowflake equivalent: CREATE DYNAMIC TABLE
CREATE OR REFRESH MATERIALIZED VIEW silver_orders (
  CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_amount EXPECT (amount > 0) ON VIOLATION WARN
)
COMMENT 'Validated orders'
AS SELECT 
  order_id,
  customer_id,
  amount,
  order_date,
  region,
  current_timestamp() AS processed_at
FROM bronze_orders
WHERE order_id IS NOT NULL;

-- Gold: Aggregate view
CREATE OR REFRESH MATERIALIZED VIEW gold_revenue_by_region
COMMENT 'Revenue aggregated by region and date'
AS SELECT
  region,
  order_date,
  COUNT(*) AS order_count,
  SUM(amount) AS total_revenue,
  AVG(amount) AS avg_order_value
FROM silver_orders
GROUP BY region, order_date;
*/

SELECT 'See comments above for SDP SQL syntax. Run these inside a Pipeline notebook.' AS note;

In [0]:
# ============================================================
# 4.3  LAKEFLOW JOBS: MULTI-TASK DAG ORCHESTRATION
# ============================================================
# Snowflake equivalent: Task DAGs (CREATE TASK ... AFTER parent_task)
# Databricks: Lakeflow Jobs with visual DAG, conditions, retries.

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import (
    Task, NotebookTask, PipelineTask, 
    TaskDependency, JobCluster, 
    CronSchedule, Source
)

w = WorkspaceClient()

# Define a multi-task job (like a Snowflake Task DAG)
job_config = {
    "name": "lab_etl_pipeline",
    "description": "Bronze → Silver → Gold ETL with quality checks",
    
    # Schedule (like Snowflake SCHEDULE = 'USING CRON 0 8 * * * UTC')
    "schedule": {
        "quartz_cron_expression": "0 0 8 * * ?",  # Daily at 8 AM
        "timezone_id": "UTC"
    },
    
    # Job cluster (ephemeral; like having a dedicated warehouse per task run)
    "job_clusters": [{
        "job_cluster_key": "etl_cluster",
        "new_cluster": {
            "spark_version": "17.3.x-scala2.12",
            "node_type_id": "i3.xlarge",
            "autoscale": {"min_workers": 1, "max_workers": 4},
            "aws_attributes": {"availability": "SPOT_WITH_FALLBACK"}
        }
    }],
    
    # Tasks with dependencies (like AFTER in Snowflake tasks)
    "tasks": [
        {
            "task_key": "ingest_raw",
            "description": "Bronze: Ingest raw data",
            "notebook_task": {
                "notebook_path": "/Users/<YOUR_USER>/etl/01_ingest",
                "source": "WORKSPACE"
            },
            "job_cluster_key": "etl_cluster"
        },
        {
            "task_key": "transform_silver",
            "description": "Silver: Clean and validate",
            "depends_on": [{"task_key": "ingest_raw"}],  # Like AFTER ingest_raw
            "notebook_task": {
                "notebook_path": "/Users/<YOUR_USER>/etl/02_transform",
                "source": "WORKSPACE"
            },
            "job_cluster_key": "etl_cluster"
        },
        {
            "task_key": "aggregate_gold",
            "description": "Gold: Business aggregates",
            "depends_on": [{"task_key": "transform_silver"}],
            "notebook_task": {
                "notebook_path": "/Users/<YOUR_USER>/etl/03_aggregate",
                "source": "WORKSPACE"
            },
            "job_cluster_key": "etl_cluster"
        },
        {
            "task_key": "run_quality_checks",
            "description": "Data quality validation",
            "depends_on": [{"task_key": "aggregate_gold"}],
            "notebook_task": {
                "notebook_path": "/Users/<YOUR_USER>/etl/04_quality",
                "source": "WORKSPACE"
            },
            "job_cluster_key": "etl_cluster"
        }
    ],
    
    # Notifications (like Snowflake ERROR_INTEGRATION)
    "email_notifications": {
        "on_failure": ["team@company.com"]
    },
    
    # Retry policy (no Snowflake equivalent)
    "max_retries": 2,
    "timeout_seconds": 3600  # 1 hour max
}

print("Job configuration defined. To create:")
print("  job = w.jobs.create(**job_config)")
print(f"\nTasks in DAG: {' → '.join([t['task_key'] for t in job_config['tasks']])}")

# Section 5 — Delta Sharing: Cross-Platform Data Distribution

## Snowflake vs Delta Sharing

| Snowflake Secure Sharing | Delta Sharing | Notes |
|---|---|---|
| Share (within Snowflake) | **Share** (cross-platform!) | Works with Spark, pandas, Power BI, Tableau, Snowflake etc. |
| Data Exchange / Marketplace | **Databricks Marketplace** | Public or private data products |
| Reader Account (free consumers) | **Open Sharing** (no Databricks needed) | Consumers use any Delta Sharing client |
| Listing (metadata + access) | **Recipient** + **Share** + **Provider** | Three-tier model |
| `CREATE SHARE` | `CREATE SHARE` | Similar SQL syntax |
| `ALTER SHARE ADD TABLE` | `ALTER SHARE ADD TABLE` | Identical pattern |
| `CREATE ACCOUNT` for consumer | `CREATE RECIPIENT` | Generates activation link |

> **Key Insight:** Delta Sharing is an **open protocol**. Unlike Snowflake sharing (Snowflake-to-Snowflake only),  
> Delta Sharing works with ANY client: pandas, Spark, Trino, Power BI, dbt, or even Snowflake itself.

In [0]:
%sql
-- ============================================================
-- 5.1  CREATE A SHARE (Equivalent to Snowflake CREATE SHARE)
-- ============================================================
-- In Snowflake: CREATE SHARE analytics_share;
-- In Databricks: Same syntax!

-- Create a share object
CREATE SHARE IF NOT EXISTS lab_analytics_share
  COMMENT 'Curated analytics data for external partners';

-- Add tables to the share (like Snowflake ALTER SHARE ... ADD TABLE)
ALTER SHARE lab_analytics_share
  ADD TABLE lab_snowflake_admin.governance_lab.events;

-- Add a schema (shares all current and future tables)
--ALTER SHARE lab_analytics_share
  --ADD SCHEMA lab_snowflake_admin.governance_lab;

-- View share contents
SHOW ALL IN SHARE lab_analytics_share;

In [0]:
%sql
-- ============================================================
-- 5.2  CREATE RECIPIENTS (External Consumers)
-- ============================================================
-- Snowflake equivalent: CREATE MANAGED ACCOUNT or add to a listing
-- Databricks: CREATE RECIPIENT generates an activation link.

-- Option A: Open sharing (recipient gets a token; no Databricks needed)
-- Like Snowflake Reader Accounts but cross-platform
CREATE RECIPIENT IF NOT EXISTS partner_acme
  COMMENT 'Acme Corp - external analytics partner';

-- Get the activation link (send this to the recipient)
-- They use it to download credentials for the Delta Sharing client
DESCRIBE RECIPIENT partner_acme;

-- Option B: Databricks-to-Databricks sharing (via sharing identifier)
-- Like Snowflake account-to-account sharing
/*
CREATE RECIPIENT IF NOT EXISTS internal_team_b
  USING ID '<recipient-metastore-sharing-identifier>';
*/

-- Grant the share to the recipient
GRANT SELECT ON SHARE lab_analytics_share TO RECIPIENT partner_acme;

-- View all recipients
SHOW RECIPIENTS;

In [0]:
# ============================================================
# 5.3  CONSUMING SHARED DATA (Recipient/Client Side)
# ============================================================
# This is what the EXTERNAL consumer does with the activation link.
# They don't need Databricks! Just pip install delta-sharing.

consumer_code = '''
# --- ON THE CONSUMER SIDE (any Python environment) ---
# pip install delta-sharing

import delta_sharing

# Path to the profile file (downloaded from activation link)
profile_file = "config.share"  # Contains endpoint URL + bearer token

# List available shares
client = delta_sharing.SharingClient(profile_file)
print(client.list_all_tables())

# Read a shared table into pandas (no Spark needed!)
df = delta_sharing.load_as_pandas(
    f"{profile_file}#lab_analytics_share.governance_lab.events"
)
print(df.head())

# Or read into Spark DataFrame
# df = delta_sharing.load_as_spark(
#     f"{profile_file}#lab_analytics_share.governance_lab.events"
# )
'''

print(consumer_code)
print("\n" + "="*70)
print("IMPORTANT: Delta Sharing is an OPEN PROTOCOL.")
print("Consumers can use: Python, Spark, Power BI, Tableau, R, Go, Rust...")
print("They do NOT need a Databricks account (unlike Snowflake sharing).")
print("="*70)

In [0]:
%sql
-- ============================================================
-- 5.4  MANAGE SHARES (Revoke, Audit, Clean Up)
-- ============================================================

-- Revoke access (like Snowflake: REVOKE ... FROM SHARE)
REVOKE SELECT ON SHARE lab_analytics_share FROM RECIPIENT partner_acme;

-- Remove a table from the share
ALTER SHARE lab_analytics_share
  REMOVE TABLE lab_snowflake_admin.governance_lab.events;

-- Audit: who accessed what (like Snowflake DATA_SHARING_USAGE)
SELECT *
FROM system.access.audit
WHERE action_name LIKE '%Share%' OR action_name LIKE '%Recipient%'
ORDER BY event_time DESC
LIMIT 20;

-- Clean up lab resources
-- DROP RECIPIENT partner_acme;
-- DROP SHARE lab_analytics_share;

# Section 6 — Declarative Automation Bundles (DABs): Infrastructure as Code

> **Key Insight:** DABs are Databricks' native IaC tool. They package notebooks, pipelines, jobs,  
> and SQL into a deployable bundle with environment-specific targets. Think of it as a  
> purpose-built alternative to Terraform that understands Databricks objects natively.  
> You can also use the Databricks Terraform provider for existing Terraform workflows.

In [0]:
# ============================================================
# 6.1  DECLARATIVE AUTOMATION BUNDLES (DABs) - PROJECT STRUCTURE
# ============================================================
# DABs define your entire deployment as code in a Git repo.
# This is how production teams deploy to Databricks.

project_structure = """
my-etl-project/                     # Git repository root
├── databricks.yml                   # Main bundle configuration
├── resources/
│   ├── jobs.yml                     # Job definitions
│   ├── pipelines.yml                # SDP pipeline definitions
│   └── dashboards.yml               # Dashboard definitions
├── src/
│   ├── bronze_ingest.py             # Pipeline notebook: bronze layer
│   ├── silver_transform.py          # Pipeline notebook: silver layer
│   ├── gold_aggregate.py            # Pipeline notebook: gold layer
│   └── quality_checks.sql           # SQL quality validations
├── tests/
│   └── test_transforms.py           # Unit tests
└── fixtures/
    └── sample_data.json             # Test data
"""

print(project_structure)
print("\nTo initialize a new DAB project:")
print("  databricks bundle init")
print("\nTo deploy:")
print("  databricks bundle validate          # Check syntax")
print("  databricks bundle deploy -t dev      # Deploy to dev target")
print("  databricks bundle deploy -t prod     # Deploy to production")
print("  databricks bundle run -t dev my_job  # Run a specific job")
print("  databricks bundle destroy -t dev     # Tear down dev resources")

In [0]:
# ============================================================
# 6.2  databricks.yml - MAIN BUNDLE CONFIGURATION
# ============================================================
# This is the heart of a DAB. Similar to dbt_project.yml or
# Snowflake's Terraform main.tf but purpose-built for Databricks.

databricks_yml = '''
# databricks.yml - Declarative Automation Bundle
bundle:
  name: etl_analytics_pipeline
  uuid: <auto-generated-on-init>

# Variables (like Terraform variables or Snowflake session params)
variables:
  catalog:
    description: "Target Unity Catalog"
    default: lab_snowflake_admin
  schema:
    description: "Target schema"
    default: governance_lab
  warehouse_id:
    description: "SQL Warehouse for queries"

# Include modular resource files
include:
  - resources/*.yml

# Targets = Environments (like Snowflake dev/prod databases)
targets:
  dev:
    mode: development              # Development mode: prefixes resources with user name
    default: true
    workspace:
      host: https://your-workspace.cloud.databricks.com
    variables:
      catalog: lab_snowflake_admin_dev
      schema: governance_lab_dev

  staging:
    workspace:
      host: https://your-workspace.cloud.databricks.com
    variables:
      catalog: lab_snowflake_admin_staging
      schema: governance_lab

  prod:
    mode: production               # Production mode: strict permissions, no prefix
    workspace:
      host: https://your-workspace.cloud.databricks.com
    variables:
      catalog: lab_snowflake_admin
      schema: governance_lab
    run_as:
      service_principal_name: "etl-prod-sp"  # Like Snowflake EXECUTE AS
'''

print(databricks_yml)

In [0]:
# ============================================================
# 6.3  resources/pipelines.yml - SDP PIPELINE DEFINITION
# ============================================================
# Define your Spark Declarative Pipeline as code.
# Equivalent to deploying Snowflake Dynamic Tables + Streams via Terraform.

pipelines_yml = '''
# resources/pipelines.yml
resources:
  pipelines:
    etl_pipeline:
      name: "${bundle.name}_etl"
      catalog: ${var.catalog}           # Unity Catalog target
      target: ${var.schema}             # Schema for output tables
      
      # Pipeline notebooks (Bronze → Silver → Gold)
      libraries:
        - notebook:
            path: ../src/bronze_ingest.py
        - notebook:
            path: ../src/silver_transform.py
        - notebook:
            path: ../src/gold_aggregate.py
      
      # Compute configuration
      clusters:
        - label: default
          autoscale:
            min_workers: 1
            max_workers: 4
            mode: ENHANCED        # Intelligent auto-scaling
      
      # Pipeline settings
      continuous: false           # Triggered mode (like Snowflake SCHEDULE)
      development: true           # Dev mode: relaxed error handling
      edition: ADVANCED           # Enables expectations (DQ checks)
      
      channel: CURRENT            # Use current runtime version
      
      # Notifications
      notifications:
        - on_failure:
            - team@company.com
          alerts:
            - on-flow-failure
'''

print(pipelines_yml)

In [0]:
# ============================================================
# 6.4  resources/jobs.yml - LAKEFLOW JOB (SCHEDULED DAG)
# ============================================================
# Define your orchestration job as code.
# Equivalent to Snowflake Task DAGs deployed via Terraform.

jobs_yml = '''
# resources/jobs.yml
resources:
  jobs:
    daily_etl_job:
      name: "${bundle.name}_daily_etl"
      description: "Daily ETL: ingest, transform, aggregate, validate"
      
      # Schedule (like Snowflake SCHEDULE = \'USING CRON ...\')  
      schedule:
        quartz_cron_expression: "0 0 8 * * ?"  # Daily at 8 AM UTC
        timezone_id: UTC
      
      # Ephemeral job cluster (like a warehouse that only exists during the job)
      job_clusters:
        - job_cluster_key: etl_cluster
          new_cluster:
            spark_version: "17.3.x-scala2.12"
            node_type_id: i3.xlarge
            autoscale:
              min_workers: 1
              max_workers: 8
            aws_attributes:
              availability: SPOT_WITH_FALLBACK
      
      # Task DAG (like Snowflake TASK ... AFTER parent_task)
      tasks:
        - task_key: run_pipeline
          pipeline_task:
            pipeline_id: ${resources.pipelines.etl_pipeline.id}
            full_refresh: false  # Incremental (like Snowflake stream processing)
        
        - task_key: quality_checks
          depends_on:
            - task_key: run_pipeline
          sql_task:
            warehouse_id: ${var.warehouse_id}
            file:
              path: ../src/quality_checks.sql
        
        - task_key: notify_downstream
          depends_on:
            - task_key: quality_checks
          notebook_task:
            notebook_path: ../src/notify.py
          job_cluster_key: etl_cluster
      
      # Error handling
      max_retries: 2
      timeout_seconds: 7200
      
      # Alerts (like Snowflake ERROR_INTEGRATION)
      email_notifications:
        on_failure:
          - oncall@company.com
        on_success:
          - team@company.com
      
      # Tags for cost tracking
      tags:
        team: data-engineering
        cost_center: DE-001
        environment: ${bundle.target}
'''

print(jobs_yml)

In [0]:
# ============================================================
# 6.5  DAB CLI COMMANDS (Deployment Workflow)
# ============================================================
# These commands run from your local terminal or CI/CD pipeline.
# Equivalent to: terraform plan / terraform apply / terraform destroy

commands = """
# =============================================
# DECLARATIVE AUTOMATION BUNDLES - CLI WORKFLOW
# =============================================

# 1. INITIALIZE a new project (like `terraform init`)
#    Creates the project skeleton with databricks.yml
databricks bundle init

# 2. VALIDATE configuration (like `terraform validate`)
#    Checks syntax, references, and permissions
databricks bundle validate -t dev

# 3. DEPLOY to a target environment (like `terraform apply`)
#    Creates/updates all resources in the workspace
databricks bundle deploy -t dev       # Development (prefixed with your username)
databricks bundle deploy -t staging   # Staging
databricks bundle deploy -t prod      # Production (requires run_as SP)

# 4. RUN a specific resource (trigger a job or pipeline)
databricks bundle run -t dev daily_etl_job
databricks bundle run -t dev etl_pipeline --full-refresh

# 5. DESTROY resources (like `terraform destroy`)
databricks bundle destroy -t dev

# =============================================
# CI/CD INTEGRATION (GitHub Actions example)
# =============================================
# In your .github/workflows/deploy.yml:
#
# jobs:
#   deploy:
#     runs-on: ubuntu-latest
#     steps:
#       - uses: actions/checkout@v4
#       - uses: databricks/setup-cli@main
#       - run: databricks bundle deploy -t prod
#         env:
#           DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
#           DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
"""

print(commands)

# Section 7 — Lab Cleanup & Quick Reference

## What We Covered

| Section | Concept | Databricks Solution |
|---|---|---|
| 1. Unity Catalog | Roles + Grants + Masking Policies | Groups + Grants + ABAC (tag-driven, inherited) |
| 2. Compute | Compute | SQL Warehouses (serverless) + Job Clusters (ephemeral) |
| 3. Managed Table | Optimization | Liquid Clustering (`CLUSTER BY AUTO`) + Predictive Optimization |
| 4. Data engineering | Pipelines and orchestration | Lakeflow Jobs + Spark Declarative Pipelines |
| 5. Data Sharing | Secure Data Sharing | Delta Sharing (open protocol, cross-platform) |
| 6. DABs | IaC | Declarative Automation Bundles (native IaC) |


In [0]:
%sql
-- ============================================================
-- 7.1  CLEANUP LAB RESOURCES
-- ============================================================
-- Uncomment to tear down all lab objects.
-- In production, use: databricks bundle destroy -t dev

/*
-- Remove ABAC policies first (required before dropping catalog)
DROP POLICY IF EXISTS region_row_filter ON CATALOG lab_snowflake_admin;
DROP POLICY IF EXISTS ssn_mask_policy ON CATALOG lab_snowflake_admin;
DROP POLICY IF EXISTS email_mask_policy ON CATALOG lab_snowflake_admin;

-- Remove manual row filter / column mask
ALTER TABLE lab_snowflake_admin.governance_lab.orders DROP ROW FILTER;
ALTER TABLE lab_snowflake_admin.governance_lab.orders ALTER COLUMN customer_email DROP MASK;

-- Remove Delta Sharing objects
DROP RECIPIENT IF EXISTS partner_acme;
DROP SHARE IF EXISTS lab_analytics_share;

-- Drop schema and all tables
DROP SCHEMA IF EXISTS lab_snowflake_admin.governance_lab CASCADE;

-- Drop the catalog
DROP CATALOG IF EXISTS lab_snowflake_admin CASCADE;
*/

SELECT 'Lab cleanup commands are commented out. Uncomment to execute.' AS status;

## Further Reading & Resources

| Topic | Documentation |
|---|---|
| Unity Catalog | https://docs.databricks.com/en/data-governance/unity-catalog/index.html |
| ABAC Policies | https://docs.databricks.com/en/data-governance/unity-catalog/abac/index.html |
| Liquid Clustering | https://docs.databricks.com/en/delta/clustering.html |
| Predictive Optimization | https://docs.databricks.com/en/optimizations/predictive-optimization.html |
| Spark Declarative Pipelines | https://docs.databricks.com/en/delta-live-tables/index.html |
| Lakeflow Jobs | https://docs.databricks.com/en/workflows/index.html |
| Delta Sharing | https://docs.databricks.com/en/delta-sharing/index.html |
| Declarative Automation Bundles | https://docs.databricks.com/en/dev-tools/bundles/index.html |
| Databricks CLI | https://docs.databricks.com/en/dev-tools/cli/index.html |
| System Tables (billing/audit) | https://docs.databricks.com/en/administration-guide/system-tables/index.html |